# 4. Sample translation and COMET

Translate ten Ukrainian reference triplets into English with Codex, retaining
both language versions and the source IDs. The sample contains five distinct
cases per actor, ordered by ID. These recorded English sentences were produced
by the assistant for this sample.

The exact underlying Codex model revision was not exposed. The recorded text
supports exact replay. COMET scores are translation-quality estimates; they do
not mark human annotation complete.

In [ ]:
import json
import tempfile
from pathlib import Path

from IPython.display import display
from pandas import DataFrame

from fairforget.dataset_workflow import find_project, load_inputs
from fairforget.io import write_json

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = Path(tempfile.mkdtemp(prefix="warbias-dataset-"))

In [ ]:
from fairforget.dataset_workflow import record_translations

sample = json.loads((project / "protocol/translation_sample.json").read_text())
translator = json.loads((project / sample["translator_config"]).read_text())
ukrainian = inputs["translation_sample_uk"]
english = record_translations(
    project,
    ukrainian,
    inputs["translation_sample_en"],
    work / "translation",
    config=translator,
    source_origin=sample["source_origin"],
)
display(
    DataFrame(
        [
            {"id": left["id"], "ukrainian": left["stereotype"], "english": right["stereotype"]}
            for left, right in zip(ukrainian, english)
        ]
    )
)
translator

In [ ]:
from fairforget.dataset_workflow import load_recorded_quality
from fairforget.translation.quality import load_comet_scorer, prepare_quality, score_quality

plan = prepare_quality(
    ukrainian,
    english,
    input_origin=sample["quality_input_origin"],
    verify_translation_bindings=True,
)
quality_config = json.loads((project / sample["quality_config"]).read_text())
report = load_recorded_quality(project / sample["quality_result"], plan)

RUN_COMET = False
prediction_options = quality_config["prediction_options"]
if RUN_COMET:
    scorer, identity = load_comet_scorer(
        quality_config["model_id"], revision=quality_config["revision"]
    )
    report = score_quality(
        plan,
        scorer=scorer,
        model_identity=identity,
        prediction_options=prediction_options,
    )
write_json(work / "quality-plan.json", plan)
write_json(work / "quality-result.json", report)
display(DataFrame(report["by_field"]).T)
result = {
    "stage": "sample_translation_comet",
    "translation_model": translator["model"],
    "coverage": report["coverage"],
    "summary": report["summary"],
    "comet_recomputed": RUN_COMET,
}
result

The default view loads the actual measured scores after checking their source
bindings. Set `RUN_COMET = True` to calculate them again with the pinned metric
model. For Apple MPS, set `prediction_options` to `{"batch_size": 8, "gpus": 1,
"accelerator": "mps", "num_workers": 1}`.

COMET QE20 uses a raw learned scale, including possible negative scores. No
acceptance cutoff is applied. Inspect the lowest scores for translation issues.

In [ ]:
display(
    DataFrame(sorted(report["segments"], key=lambda row: row["score"]))[
        ["id", "field", "src", "mt", "score"]
    ]
)